Livrable 1 : Classification binaire

Etape 1 :

- Téléchargement du dataset
- Redimensionnement des images en `180 x 180`.
- Découpage en jeu d’entraînement et jeu de test.
- Optimisation du pipeline avec `cache()` et `prefetch()` pour accélérer l’apprentissage.

In [1]:
%pip install tensorflow
%pip install numpy
%pip install os
%pip install PIL
%pip install matplotlib

import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
import tensorflow as tf

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /home/tt/Documents/Bloc_IA/Prosit_1/.venv/bin/python -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /home/tt/Documents/Bloc_IA/Prosit_1/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /home/tt/Documents/Bloc_IA/Prosit_1/.venv/bin/python -m pip install --upgrade pip
ERROR: No matching distribution found for os
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not fi

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

In [ ]:
import os
import tensorflow as tf

data_dir = "Dataset"
deleted = 0

print("Début du scan et du nettoyage des images...")

for subdir in os.listdir(data_dir):
    subdir_path = os.path.join(data_dir, subdir)
    if os.path.isdir(subdir_path):
        for file in os.listdir(subdir_path):
            filepath = os.path.join(subdir_path, file)
            try:
                # Tente de décoder l'image avec la même fonction TensorFlow utilisée par le Dataset
                image_string = tf.io.read_file(filepath)
                _ = tf.image.decode_image(image_string)
            except Exception as e:
                print(f"Fichier corrompu ou invalide détecté : {filepath}")
                try:
                    os.remove(filepath)
                    deleted += 1
                    print("-> Fichier supprimé avec succès.")
                except PermissionError:
                    print(f"-> Impossible de supprimer {filepath} : Droits administrateur requis.")
                except Exception as e_rm:
                    print(f"-> Erreur lors de la suppression de {filepath} : {e_rm}")

print(f"Nettoyage terminé. {deleted} images corrompues supprimées.")

In [ ]:
batch_size = 32
img_height = 180
img_width = 180
data_dir = "Dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

class_names = train_ds.class_names
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Etape 2:

- Visualisation de plusieurs images pour comprendre la variété des données.
- Observation des difficultés : angles différents, luminosité variable, arrière-plans variés, présence d’objets parasites, Perte d'information critique avec le redimensionnement, Le biais lié à la couleur, Le bruit et les artefacts.


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()

Etape 3 :

Construction d’un premier CNN
Le modèle construit comprend :

une normalisation des pixels avec Rescaling(1./255) ;
trois blocs convolutifs successifs :
Conv2D(16, 3x3) + MaxPooling2D,
Conv2D(32, 3x3) + MaxPooling2D,
Conv2D(64, 3x3) + MaxPooling2D ;
une couche Flatten ;
une couche dense de 128 neurones ;
une couche de sortie softmax pour classer les 6 datasets

In [ ]:
num_classes = len(class_names) # 6 datasets

model = Sequential([
  layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

model.summary()

### Explication des résultats obtenus jusqu’ici

- Optimiseur : `adam`
- Perte : `SparseCategoricalCrossentropy`
- Métrique : `accuracy`

Etape 4 : 

Les courbes d’accuracy et de loss permettent d’analyser les performances.

In [ ]:
epochs = 10

history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()